# 🐼 Pandas Library — Data Manipulation & Analysis
### Complete Beginner-Friendly Notes
---
> **Topics Covered:** Reading CSVs • Exploring Data • Missing Values • Renaming Columns • Type Casting • Lambda Apply • GroupBy & Aggregation • Merging & Joining DataFrames

---

## 📌 What is Pandas?

**Pandas** is a Python library used for **data manipulation and analysis**.

Think of it like **Excel, but on steroids** — you can load, clean, transform, and analyze data with just a few lines of Python.

### Key Use Cases:
- **EDA (Exploratory Data Analysis)** — understand your data
- **Feature Engineering** — transform raw data for ML models
- **Data Cleaning** — handle missing, wrong, or duplicate data

### Core Data Structures:
| Structure | Description | Like... |
|-----------|-------------|--------|
| `Series` | 1D labeled array | A single Excel column |
| `DataFrame` | 2D labeled table | An entire Excel sheet |


---
## 📂 SECTION 1: Importing Pandas & Reading CSV


In [ ]:
# Step 1: Import pandas
import pandas as pd

# Step 2: Read a CSV file into a DataFrame
# Make sure 'data.csv' is in the same folder as this notebook
df = pd.read_csv('data.csv')

# 'df' is now a DataFrame — a table that holds your data
print("✅ CSV Loaded Successfully!")

### 💡 What just happened?
- `pd.read_csv('data.csv')` reads the file and stores it as a **DataFrame** object in variable `df`
- Columns in the dataset: `date`, `category`, `value`, `product`, `sales`, `region`

---

## 🔍 SECTION 2: Exploring Your Data
Before doing anything, always **look at** your data first.


In [ ]:
# First 5 rows
df.head(5)

In [ ]:
# Last 5 rows
df.tail(5)

In [ ]:
# Statistical Summary (only for numerical columns)
# Shows: count, mean, std, min, 25%, 50%, 75%, max
df.describe()

In [ ]:
# Data types of each column
# 'object' = string, 'float64' = decimal number, 'int64' = whole number
df.dtypes

### 📊 Quick Reference Table

| Method | What it does |
|--------|-------------|
| `df.head(n)` | First n rows (default 5) |
| `df.tail(n)` | Last n rows (default 5) |
| `df.describe()` | Stats: mean, std, min, max, percentiles |
| `df.dtypes` | Data type of each column |
| `df.shape` | (rows, columns) |
| `df.columns` | List all column names |
| `df.info()` | Overview: types + non-null counts |

---

## 🕳️ SECTION 3: Handling Missing Values

Missing values = cells with **no data** → shown as `NaN` (Not a Number) in Pandas.

Real-world data is always dirty. Learning to handle NaN is essential!


In [ ]:
# Step 1: See True/False for every cell (True = missing)
df.isnull()

In [ ]:
# Step 2: Check which COLUMNS have missing values
# axis=0 → column-wise (default)
df.isnull().any(axis=0)

In [ ]:
# Step 3: COUNT how many missing values per column
df.isnull().sum()

In [ ]:
# Strategy 1: Fill ALL missing values with 0
# ⚠️ Warning: Don't modify original df — use a copy
df_filled_zero = df.fillna(0)
df_filled_zero.head()

In [ ]:
# Strategy 2 (BETTER): Fill with MEAN of that column
# Why mean? Because it doesn't distort the data distribution

# Creating a new column 'sales_fillna' with NaN replaced by mean of 'sales'
df['sales_fillna'] = df['sales'].fillna(df['sales'].mean())

print("Mean of sales column:", df['sales'].mean())
df[['sales', 'sales_fillna']].head(10)

### 💡 Which Strategy to Use?

| Strategy | When to Use |
|----------|-------------|
| `fillna(0)` | Count-based data, or when 0 makes logical sense |
| `fillna(mean)` | Numerical columns with normal distribution |
| `fillna(median)` | Numerical columns with outliers/skewed data |
| `fillna(mode)` | Categorical columns |
| `dropna()` | Drop rows with missing values (use carefully!) |

---

## ✏️ SECTION 4: Renaming Columns

Sometimes column names are confusing or too long. Rename them!


In [ ]:
# Rename 'date' column to 'sales_date'
# columns parameter takes a DICTIONARY: {old_name: new_name}
# inplace=True modifies the original df directly

df = df.rename(columns={'date': 'sales_date'})
df.head(3)

In [ ]:
# Rename MULTIPLE columns at once
df = df.rename(columns={
    'sales_date': 'date',      # rename back
    'category': 'cat',
    'region': 'zone'
})
df.head(3)

### 💡 Key Point
The `columns` parameter takes a **dictionary** format: `{old_name: new_name}`

---

## 🔄 SECTION 5: Changing Data Types (Type Casting)

When you load a CSV, Pandas guesses the type. Sometimes it guesses wrong! You can fix it.


In [ ]:
# Check current data types
df.dtypes

In [ ]:
# Convert 'value' column from float to int
# First fill NaN values, then convert

df['value_new'] = df['value'].fillna(df['value'].mean()).astype(int)
print("New column dtype:", df['value_new'].dtype)
df[['value', 'value_new']].head()

### ⚠️ Why fill NaN BEFORE converting to int?
Because `int` type **cannot store NaN** (NaN is a float concept). If you try to convert a column with NaN to int directly, Python will throw an error!

### Common Data Types in Pandas:
| Pandas dtype | Meaning |
|-------------|--------|
| `object` | String / text |
| `int64` | Whole numbers |
| `float64` | Decimal numbers |
| `bool` | True / False |
| `datetime64` | Date and time |

---

## ⚡ SECTION 6: Applying Functions with .apply() and Lambda

Sometimes you want to transform each value in a column using a formula or custom logic.


In [ ]:
# Example: Double the 'value' column
# Lambda is an anonymous (one-line) function
# lambda x: x*2  means → take each value x, return x*2

df['value_doubled'] = df['value'].apply(lambda x: x * 2)
df[['value', 'value_doubled']].head()

In [ ]:
# Example: Apply a CUSTOM function
def apply_discount(price):
    """Apply 10% discount to price"""
    if price > 50:
        return price * 0.9  # 10% off
    else:
        return price

df['discounted_value'] = df['value'].apply(apply_discount)
df[['value', 'discounted_value']].head()

### 💡 Lambda Syntax Simplified
```python
lambda x: <expression>
# is same as:
def func(x):
    return <expression>
```
Use **lambda** for simple one-liners. Use **def** for complex logic.

---

## 📊 SECTION 7: Data Aggregation & GroupBy

GroupBy = **Split → Apply → Combine**

Like SQL's `GROUP BY`. Split data into groups, apply a function, combine results.


In [ ]:
# Example 1: Mean of 'value' grouped by 'product'
grouped_mean = df.groupby('product')['value'].mean()
print(grouped_mean)

In [ ]:
# Example 2: GroupBy on TWO columns — Sum of value by product AND region
grouped_sum = df.groupby(['product', 'region'])['value'].sum()
print(grouped_sum)

In [ ]:
# Example 3: Mean by product + region
grouped_mean2 = df.groupby(['product', 'region'])['value'].mean()
print(grouped_mean2)

In [ ]:
# Example 4: MULTIPLE aggregate functions at once using .agg()
grouped_agg = df.groupby('region')['value'].agg(['mean', 'sum', 'count'])
print(grouped_agg)

### 📌 Common Aggregation Functions
| Function | What it does |
|----------|-------------|
| `.mean()` | Average |
| `.sum()` | Total |
| `.count()` | Number of non-null values |
| `.min()` | Minimum value |
| `.max()` | Maximum value |
| `.std()` | Standard deviation |
| `.agg([...])` | Multiple functions at once |

---

## 🔗 SECTION 8: Merging & Joining DataFrames

Just like SQL JOINs — combine two tables based on a common column (key).


In [ ]:
import pandas as pd

# Create two sample DataFrames
df1 = pd.DataFrame({'key': ['A', 'B', 'C'], 'value1': [1, 2, 3]})
df2 = pd.DataFrame({'key': ['A', 'B', 'D'], 'value2': [4, 5, 6]})

print("DataFrame 1:")
print(df1)
print("\nDataFrame 2:")
print(df2)

In [ ]:
# INNER JOIN — only matching keys from BOTH tables
inner = pd.merge(df1, df2, on='key', how='inner')
print("INNER JOIN:")
print(inner)

In [ ]:
# OUTER JOIN — all keys from BOTH tables (NaN if no match)
outer = pd.merge(df1, df2, on='key', how='outer')
print("OUTER JOIN:")
print(outer)

In [ ]:
# LEFT JOIN — all keys from LEFT table, match from right (NaN if no match)
left = pd.merge(df1, df2, on='key', how='left')
print("LEFT JOIN:")
print(left)

In [ ]:
# RIGHT JOIN — all keys from RIGHT table, match from left (NaN if no match)
right = pd.merge(df1, df2, on='key', how='right')
print("RIGHT JOIN:")
print(right)

### 🔗 Join Types — Visual Summary

| Join Type | Keeps Rows From | NaN Appears When |
|-----------|----------------|------------------|
| `inner` | Both tables (only matches) | — |
| `outer` | Both tables (all rows) | No match on either side |
| `left` | All of left + matching right | Right has no match |
| `right` | All of right + matching left | Left has no match |

---

---
# 🚀 QUICK REVISION — Summary Sheet


## 📋 Pandas Cheatsheet

```python
import pandas as pd

# ── LOADING ──────────────────────────────────────────
df = pd.read_csv('file.csv')         # Read CSV
df = pd.read_excel('file.xlsx')      # Read Excel

# ── EXPLORING ────────────────────────────────────────
df.head(5)                           # First 5 rows
df.tail(5)                           # Last 5 rows
df.describe()                        # Stats summary
df.dtypes                            # Column data types
df.shape                             # (rows, cols)
df.info()                            # Full overview
df.columns                           # Column names

# ── MISSING VALUES ───────────────────────────────────
df.isnull()                          # True/False per cell
df.isnull().any(axis=0)              # Any NaN per column?
df.isnull().sum()                    # Count NaN per column
df.fillna(0)                         # Fill NaN with 0
df['col'].fillna(df['col'].mean())   # Fill NaN with mean
df.dropna()                          # Drop rows with NaN

# ── RENAME & TYPE CAST ───────────────────────────────
df.rename(columns={'old': 'new'})    # Rename column
df['col'].astype(int)                # Change dtype

# ── APPLY & LAMBDA ───────────────────────────────────
df['col'].apply(lambda x: x * 2)    # Apply function
df['col'].apply(my_function)         # Apply custom func

# ── GROUPBY & AGGREGATION ────────────────────────────
df.groupby('col')['val'].mean()      # GroupBy + mean
df.groupby(['c1','c2'])['val'].sum() # Multi-GroupBy
df.groupby('col')['val'].agg(['mean','sum','count'])

# ── MERGING ──────────────────────────────────────────
pd.merge(df1, df2, on='key', how='inner')   # Inner join
pd.merge(df1, df2, on='key', how='outer')   # Outer join
pd.merge(df1, df2, on='key', how='left')    # Left join
pd.merge(df1, df2, on='key', how='right')   # Right join
```


---
# ❓ Important Questions & Answers
*(Frequently Asked — Must Know!)*


**Q1. What is the difference between `df.head()` and `df.tail()`?**
> `df.head(n)` returns the **first n rows** of the DataFrame. `df.tail(n)` returns the **last n rows**. Default n = 5.

**Q2. What is `NaN` in Pandas?**
> NaN stands for **Not a Number**. It represents a missing or undefined value in a DataFrame. It is a float concept, so integer columns cannot hold NaN directly.

**Q3. How do you count missing values per column?**
> `df.isnull().sum()` — This returns the count of NaN values in each column.

**Q4. Why should we fill missing values with mean instead of 0?**
> Filling with 0 can distort data if 0 is not a valid value. Mean keeps the **statistical distribution** of the column intact and doesn't introduce bias.

**Q5. What does `df.describe()` show?**
> It shows statistical summary for **numeric columns**: count, mean, std (standard deviation), min, 25th percentile, 50th percentile (median), 75th percentile, and max.

**Q6. What does 'object' dtype mean in Pandas?**
> `object` dtype means the column holds **string/text** data.

**Q7. Why do we get an error when converting a column with NaN to `int`?**
> Because `int` type cannot store NaN values in standard Pandas. You must **fill NaN first** with `fillna()` before using `.astype(int)`.

**Q8. What is the difference between Inner Join and Outer Join?**
> - **Inner Join**: Returns only rows where the key **matches in BOTH** DataFrames.
> - **Outer Join**: Returns **all rows** from both DataFrames. Where there's no match, NaN is filled.

**Q9. How does `groupby()` work in Pandas?**
> It follows the **Split → Apply → Combine** strategy:
> 1. **Split** data into groups based on a column
> 2. **Apply** an aggregation function (mean, sum, count...)
> 3. **Combine** results into a new DataFrame

**Q10. What is the use of `.apply()` with lambda?**
> `.apply(lambda x: expr)` lets you apply a **custom transformation** to each element of a column. Example: `df['col'].apply(lambda x: x * 2)` doubles every value.

**Q11. How do you rename a column in Pandas?**
> `df.rename(columns={'old_name': 'new_name'})` — Pass a dictionary with old → new mappings.

**Q12. What is the difference between Left Join and Right Join?**
> - **Left Join**: All rows of the **left** DataFrame are kept. Matching rows from right are added; non-matching get NaN.
> - **Right Join**: All rows of the **right** DataFrame are kept. Same logic in reverse.

**Q13. What does `df.isnull().any(axis=1)` do?**
> `axis=1` checks **row-wise** — returns True for each row that has at least one missing value. `axis=0` is column-wise.

**Q14. Can you apply multiple aggregation functions in GroupBy?**
> Yes! Use `.agg(['mean', 'sum', 'count'])` to apply multiple functions at once.

**Q15. How do you GroupBy on multiple columns?**
> Pass a **list** of column names: `df.groupby(['product', 'region'])['value'].sum()`
